Ejecuta la siguiente celda e introduce la contraseña facilitada en el curso para desbloquear el cuaderno.


In [ ]:
%pip install -q git+https://github.com/dcrespons14/cam-estrategia-aula.git
from cam_estrategia import desbloquear_cuaderno
desbloquear_cuaderno()


# Aprendizaje por refuerzo: la estrategia de Piastri

En el cuaderno de la clase 4.6 construimos un **modelo de carrera** de México y vimos que Piastri, con el tercer mejor ritmo base, acaba casi siempre 5.º con la estrategia que usó en 2025.

En este cuaderno dejamos que Piastri **aprenda su propia estrategia** con **aprendizaje por refuerzo**: el resto de pilotos mantienen sus estrategias de 2025 y Piastri corre miles de carreras, prueba estrategias distintas y aprende cuál es la que más puntos le da.

## Inicialización

Importamos las librerías relevantes.

In [ ]:
# Librerías propias (ver src/race_analysis y src/race_models)
from race_analysis.ingestion.openf1 import get, load_race
from race_analysis.ingestion.laps import clean_data
from race_analysis.tyre_deg.pace import fit_pooled_race_pace
from race_analysis.overtaking_model.overtaking import find_overtakes, attack_situations, fit_logistic
from race_models.f1.config import RaceConfig
from race_models.f1.models import base_pace_from_race_pace, tyre_deg_from_race_pace
from race_models.f1.engine.simulator import simulate_race
from race_models.f1.engine.strategy import strategy_name
from race_models.f1.analysis.race_plots import plot_race_trace, plot_tyre_strategies
from race_models.f1.rl.env import StrategyEnv, POINTS
from race_models.f1.rl.monte_carlo import MonteCarloAgent
from race_models.f1.rl.q_learning import QLearningAgent
from race_models.f1.analysis.rl_plots import plot_learning_curve, plot_q_values

# Librerías de Python
import numpy as np
import pandas as pd
from IPython.display import display

## 1. El modelo de carrera

Repetimos el modelo de carrera del cuaderno de la clase 4.6, con los mismos datos de México 2023-2025: ritmo base, modelo lineal de neumáticos y modelo de adelantamiento.

In [ ]:
TRACK = "Mexico"
YEARS = [2023, 2024, 2025]
FUEL_EFFECT = 0.05  # s por vuelta de combustible a bordo
MAX_GAP = 1.5       # distancia máxima (s) con el coche de delante para considerar que hay un ataque

races = {year: load_race(year, TRACK) for year in YEARS}
laps = {year: clean_data(race) for year, race in races.items()}
for frame in laps.values():
    frame["fuel_corrected"] = frame["lap_time"] - FUEL_EFFECT * frame["laps_remaining"]

# Modelo de neumáticos y ritmo base
linear = fit_pooled_race_pace(pd.concat(laps.values(), ignore_index=True), model="lineal")
pace = linear[2025]
tlas = {driver["driver_number"]: driver["name_acronym"] for driver in get("drivers", session_key=races[2025].session_key)}

# Modelo de adelantamiento
overtaking_data = {year: find_overtakes(race, laps[year], max_gap=MAX_GAP) for year, race in races.items()}
attacks = pd.concat([attack_situations(races[year], laps[year], linear[year], overtaking_data[year], max_gap=MAX_GAP)
                     for year in YEARS], ignore_index=True)
overtaking_model = fit_logistic(attacks["delta_pace"], attacks["overtake"])

config = RaceConfig(base_pace=base_pace_from_race_pace(pace, tlas), tyre_deg=tyre_deg_from_race_pace(pace),
                    overtaking_lr=[overtaking_model.intercept, overtaking_model.slope])
config

## 2. La estrategia real de Piastri

Usamos las estrategias de 2025 del cuaderno de la clase 4.6. Piastri hizo dos paradas: **S - M24 - S47**.

In [ ]:
strategies_2025 = {
    "NOR": [("S", 34), ("M", 37)],
    "LEC": [("S", 31), ("M", 40)],
    "HAM": [("S", 23), ("M", 24), ("S", 24)],
    "RUS": [("S", 25), ("M", 23), ("S", 23)],
    "VER": [("M", 37), ("S", 34)],
    "ANT": [("S", 22), ("M", 25), ("S", 24)],
    "PIA": [("S", 24), ("M", 23), ("S", 24)],
    "HAD": [("M", 34), ("S", 37)],
    "BEA": [("S", 24), ("M", 24), ("S", 23)],
    "TSU": [("M", 36), ("S", 35)],
    "OCO": [("S", 27), ("M", 44)],
    "SAI": [("M", 17), ("S", 29), ("S", 25)],
    "HUL": [("S", 25), ("M", 46)],
    "ALO": [("S", 20), ("M", 51)],
    "BOR": [("M", 37), ("S", 34)],
    "ALB": [("H", 41), ("S", 30)],
    "GAS": [("M", 33), ("S", 38)],
    "STR": [("S", 26), ("M", 17), ("S", 28)],
    "COL": [("H", 48), ("S", 23)],
}
RIVALS = ["PIA", "NOR", "VER", "LEC", "HAM", "RUS", "ANT", "BEA"]

Como en la clase 4.6, para valorar una estrategia corremos **muchas carreras** y miramos la distribución de resultados.

In [ ]:
EVALUATION_SEEDS = range(100_000, 100_200)  # carreras de evaluación: el agente no las ve al entrenar


def evaluate(stints):
    """Posición final de Piastri en las 200 carreras de evaluación con una estrategia fija."""
    positions = []
    for seed in EVALUATION_SEEDS:
        race = simulate_race(config, {**strategies_2025, "PIA": stints}, seed=seed)
        positions.append(race.final_order.index("PIA") + 1)
    return pd.Series(positions)


def summary(positions):
    """Tabla con la posición y los puntos de cada estrategia, como en la clase 4.6."""
    positions = pd.DataFrame(positions)
    points = positions.apply(lambda column: column.map(POINTS).fillna(0))
    table = pd.DataFrame({
        "Posición media": positions.mean(),
        "Puntos medios": points.mean(),
        "Puntos STD": points.std(),
        "Podio": (positions <= 3).mean(),
        "Mejor": positions.min(),
        "Peor": positions.max(),
    })
    with pd.option_context("display.float_format", "{:.2f}".format):
        display(table.sort_values("Puntos medios", ascending=False))


real_positions = evaluate(strategies_2025["PIA"])
summary({"S - M24 - S47 (real)": real_positions})

## 3. La carrera como un problema de aprendizaje por refuerzo

En aprendizaje por refuerzo hay un **agente** que toma decisiones sobre cómo **actuar** según el **estado** de su entorno, y recibe una **recompensa** según lo bien que lo haya hecho. Como un perro que aprende a base de galletas.

- **Agente**: el estratega de Piastri.
- **Acción**: paradas a boxes y elección de compuesto.
- **Estado**: lo que sabe el estratega de Piastri cuando decide: la vuelta, los compuestos que ya ha usado, las paradas que ha hecho, lo que están haciendo los rivales, etc.
    
    En nuestro caso, como es un problema estático y sencillo, el estado es simplemente la vuelta, los compuestos usados y las paradas que ha hecho PIA.
- **Recompensa**: los **puntos** de la posición final.

In [ ]:
env = StrategyEnv(config, strategies_2025, driver="PIA")

state = env.reset(seed=0)
actions = env.legal_actions()

print("Estado en la salida:", state)

Corremos una carrera con la estrategia real, stint a stint. La recompensa es 0 hasta la bandera a cuadros.

In [ ]:
state = env.reset(seed=0)
for stint in strategies_2025["PIA"]:
    state, reward, done = env.step(stint)
    print(f"Stint {stint} -> estado {state}, recompensa {reward}, fin de carrera: {done}")
print(f"{env.strategy_name()}: Piastri acaba {env.position()}.º")

## 4. Entrenamiento

Piastri corre 15.000 carreras. Al final de cada una de ellas, recibe una recompensa en función de su resultado, y poco a poco va aprendiendo qué estrategias funcionan mejor.

En vez de siempre hacer lo que cree que es óptimo, va explorando en función de la certeza que tenga.

In [ ]:
SEED = 0            # cambiadla para repetir el entrenamiento con otras carreras
EPISODES = 15_000

agent = QLearningAgent(seed=SEED, exploration=10.0)
history = agent.train(env, episodes=EPISODES, seed=SEED * 1_000_000, progress=True)
history.tail()

La **curva de aprendizaje** tiene dos líneas:

- **Entrenamiento**: los puntos medios de las últimas 500 carreras, explorando.
- **Estrategia aprendida**: los puntos de la estrategia que elegiría el agente sin explorar, en cada comprobación.

In [ ]:
plot_learning_curve(history, window=500, references={"S - M24 - S47 (real)": real_positions.map(POINTS).fillna(0).mean()},
                    title="Piastri · Curva de aprendizaje");

Estas son las veces que ha cambiado la estrategia aprendida durante el entrenamiento:

In [ ]:
checks = history.dropna(subset=["learned"])
changes = checks[checks["learned"] != checks["learned"].shift()]
changes[["episode", "learned", "learned_points"]].rename(
    columns={"episode": "Carrera", "learned": "Estrategia aprendida", "learned_points": "Puntos (50 carreras)"})

In [ ]:
print("Estrategias más usadas en las últimas 2.000 carreras de entrenamiento:")
history.tail(2_000)["strategy"].value_counts().head(5)

## 5. ¿Qué ha aprendido Piastri?

Se ha dado cuenta de que hacer estrategias con un primer stint de 38-46 vueltas es el grupo de estrategias que mejor funciona. Dentro de este grupo, hacer el stint con H retorna el máximo de puntos esperados.

In [ ]:
reward, decisions = agent.play(env, seed=EVALUATION_SEEDS[0])
learned_stints = list(env.stints)
learned_name = env.strategy_name()
print(f"Estrategia aprendida: {learned_name}  {learned_stints}")

plot_q_values(agent, decisions[0][0], min_visits=5, title="Piastri · Primer stint: puntos esperados");

In [ ]:
race = env.result()
plot_race_trace(race, drivers=RIVALS, title=f"Piastri · {learned_name} (aprendida)");

## 6. ¿Es mejor que la estrategia real?

Comparamos las dos estrategias en las mismas 200 carreras de evaluación.

In [ ]:
learned_positions = evaluate(learned_stints)
summary({"S - M24 - S47 (real)": real_positions, f"{learned_name} (aprendida)": learned_positions})